In [2]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 


In [3]:
# Đọc dữ liệu từ file excel
route = pd.read_excel("dim_route.xlsx")
product_type = pd.read_excel("product_type.xlsx")
product = pd.read_excel("product.xlsx")
customer_order = pd.read_excel("customer_order.xlsx")
link = pd.read_excel("fact_link.xlsx")
warehouse = pd.read_excel("fact_warehouse.xlsx")

# **1. Route**

**1. Data understanding**

In [4]:
# Dữ liệu
print("- Dữ liệu")
route.head()

- Dữ liệu


,route_id,exchange_rate,name,note,ship_time,unit_buying_price,unit_deposit_price,difference_rate,is_update_auto,min_weight
0,8,1.55,VND - IDR,VND - IDR,2026-10-07 00:00:00,275000,275000,0.0,False,0.0
1,2,185.00,JPY,Tuyến Nhật,2026-08-07 00:00:00,145000,145000,NaN,False,0.5
2,1,28300.00,USD,Tuyến Mỹ,2026-12-10 00:00:00,235000,235000,NaN,False,1.0
3,4,18.80,KRW,Tuyến Hàn,6,135000,135000,NaN,False,1.0
4,6,1.68,IDR,Tuyến Indo,7,280000,280000,NaN,False,1.0


In [5]:
# Số dòng và cột dữ liệu
print(f"Số dòng dữ liệu: {route.shape[0]} dòng.")
print(f"Số cột dữ liệu: {route.shape[1]} cột.")


Số dòng dữ liệu: 5 dòng.
Số cột dữ liệu: 10 cột.


In [6]:
# Tổng quan thuộc tính dữ liệu
route.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   route_id            5 non-null      int64  
 1   exchange_rate       5 non-null      float64
 2   name                5 non-null      object 
 3   note                5 non-null      object 
 4   ship_time           5 non-null      object 
 5   unit_buying_price   5 non-null      int64  
 6   unit_deposit_price  5 non-null      int64  
 7   difference_rate     1 non-null      float64
 8   is_update_auto      5 non-null      bool   
 9   min_weight          5 non-null      float64
dtypes: bool(1), float64(3), int64(3), object(3)
memory usage: 497.0+ bytes


In [7]:
# Thuộc tính dạng số
numerical_cols_route = route.select_dtypes(include = ['number']).columns
print(f"- Numerical columns: {numerical_cols_route}")

- Numerical columns: Index(['route_id', 'exchange_rate', 'unit_buying_price', 'unit_deposit_price',
       'difference_rate', 'min_weight'],
      dtype='object')


In [8]:
# Thống kê thuộc tính dạng số
route.describe()

,route_id,exchange_rate,unit_buying_price,unit_deposit_price,difference_rate,min_weight
count,5.000000,5.000000,5.000000,5.000000,1.0,5.000000
mean,4.200000,5701.406000,214000.000000,214000.000000,0.0,0.700000
std,2.863564,12633.234263,69856.996786,69856.996786,NaN,0.447214
min,1.000000,1.550000,135000.000000,135000.000000,0.0,0.000000
25%,2.000000,1.680000,145000.000000,145000.000000,0.0,0.500000
50%,4.000000,18.800000,235000.000000,235000.000000,0.0,1.000000
75%,6.000000,185.000000,275000.000000,275000.000000,0.0,1.000000
max,8.000000,28300.000000,280000.000000,280000.000000,0.0,1.000000


In [9]:
# Thuộc tính dạng phân loại
categorical_cols_route = route.select_dtypes(include=['object', 'category']).columns
print(f"- Categorical columns: {categorical_cols_route}")

- Categorical columns: Index(['name', 'note', 'ship_time'], dtype='object')


In [10]:
# Thống kê thuộc tính phân loại
for col in categorical_cols_route:
    print(f"Phân bổ thuộc tính {col}:")
    print(route[col].value_counts(ascending= False))

Phân bổ thuộc tính name:
name
VND - IDR    1
JPY          1
USD          1
KRW          1
IDR          1
Name: count, dtype: int64
Phân bổ thuộc tính note:
note
VND - IDR     1
Tuyến Nhật    1
Tuyến Mỹ      1
Tuyến Hàn     1
Tuyến Indo    1
Name: count, dtype: int64
Phân bổ thuộc tính ship_time:
ship_time
2026-10-07 00:00:00    1
2026-08-07 00:00:00    1
2026-12-10 00:00:00    1
6                      1
7                      1
Name: count, dtype: int64


**2. Data cleaning**

In [11]:
# Kiểm tra giá trị null
route.isnull().sum()

route_id              0
exchange_rate         0
name                  0
note                  0
ship_time             0
unit_buying_price     0
unit_deposit_price    0
difference_rate       4
is_update_auto        0
min_weight            0
dtype: int64

In [12]:
# Kiểm tra duplicated values
print(f"-Số dòng bị trùng lặp: {route.duplicated().sum()} dòng.")

-Số dòng bị trùng lặp: 0 dòng.


**3. Outliers**

In [13]:
# Không cần thiết tìm outliers cho bảng Route

**4. Feature Engineering**

In [14]:
# Chuẩn hóa thời gian
route['ship_time']= pd.to_datetime(route['ship_time'])



In [15]:
# Giá trị đơn hàng tối thiểu
route['min_order_value'] = route['unit_deposit_price']*route['min_weight']



In [16]:
# Phân route theo khu vực
region_map = {'IDR':'Southest Asia',
              'VND - IDR':'Southest Asia',
              'JPY':'East Asia',
              'KRW':'East Asia',
              'USD':'America'}
route['region'] = route['name'].map(region_map)


In [17]:
# Tạo cột kiểm tra trạng thái lợi nhuận
def check_profit_status(x):
    if x['unit_buying_price'] < x['unit_deposit_price']:
        return 'Có lãi'
    elif x['unit_buying_price'] == x['unit_deposit_price']:
        return 'Hòa vốn'
    else: 
        return 'Lỗ vốn'

route['profit_status'] = route.apply(check_profit_status, axis = 1)


# **2. Product Type**

**1. Data Understanding**

In [18]:
# Dữ liệu
print("- Dữ liệu:")
product_type.head()

- Dữ liệu:


,product_type_id,is_fee,product_type_name
0,1,True,Điện tử
1,2,True,Nước hoa
2,3,True,Hàng hiệu
3,4,True,"Giá trị cao trên 1,000 USD"
4,5,True,Cigar


In [19]:
# Số dòng và cột dữ liệu
print(f"-Số dòng của dữ liệu: {product_type.shape[0]} dòng.")
print(f"-Số cột của dữ liệu: {product_type.shape[1]} cột.")

-Số dòng của dữ liệu: 22 dòng.
-Số cột của dữ liệu: 3 cột.


In [20]:
# Tổng quan thuộc tính dữ liệu
product_type.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   product_type_id    22 non-null     int64 
 1   is_fee             22 non-null     bool  
 2   product_type_name  22 non-null     object
dtypes: bool(1), int64(1), object(1)
memory usage: 506.0+ bytes


In [21]:
# Thuộc tính dạng số
numerical_cols_ptype = product_type.select_dtypes(include=['number']).columns
print(f"-Numerical columns: {numerical_cols_ptype}")

-Numerical columns: Index(['product_type_id'], dtype='object')


In [22]:
# Thuộc tính dạng phân loại
categorical_cols_ptype = product_type.select_dtypes(include=['object', 'category']).columns
print(f"-Categorical columns: {categorical_cols_ptype}")

-Categorical columns: Index(['product_type_name'], dtype='object')


In [23]:
# Thống kê thuộc tính dạng phân loại
for col in categorical_cols_ptype:
    print(f"Phân bổ thuộc tính {col}:")
    print(product_type[col].value_counts(ascending = False))

Phân bổ thuộc tính product_type_name:
product_type_name
Điện tử                       1
Nước hoa                      1
Hàng hiệu                     1
Giá trị cao trên 1,000 USD    1
Cigar                         1
Rượu                          1
Khác                          1
Quần Áo                       1
Giày dép                      1
Thiết Bị Xe/Phụ Tùng          1
Mỹ Phẩm                       1
Tài Chính                     1
Mẹ & Bé                       1
Đồ Chơi                       1
Thực Phẩm                     1
Thú Cưng                      1
Gia Dụng                      1
Đồ Trang Trí                  1
Thể Thao                      1
Văn Phòng Phẩm                1
Phụ Kiện/ Trang Sức           1
Thuốc/Thực Phẩm Chức Năng     1
Name: count, dtype: int64


**2.Data Cleaning**


In [24]:
# Kiểm tra giá trị null
product_type.isnull().sum()

product_type_id      0
is_fee               0
product_type_name    0
dtype: int64

In [25]:
#Kiểm tra giá trị trùng lặp
print(f"Số dòng bị trùng lặp: {product_type.duplicated().sum()}")

Số dòng bị trùng lặp: 0


**3. Outliers**

In [26]:
# Không cần thiết kiểm tra dữ liệu ngoại lai bảng product_type

**4.Feature Engineeing**

In [27]:
# Kiểm tra mức độ rủi ro
def risk_level(category):
    high_risk = [
        'Nước hoa', 'Rượu', 'Cigar', 'Hàng hiệu',
        'Giá trị cao trên 1,000 USD', 'Mỹ Phẩm',
        'Thuốc/Thực Phẩm Chức Năng', 'Điện tử'
    ]
    
    medium_risk = [
        'Thiết Bị Xe/Phụ Tùng', 'Gia Dụng',
        'Thể Thao', 'Phụ Kiện/ Trang Sức', 'Khác'
    ]
    
    if category in high_risk:
        return 'High Risk'
    elif category in medium_risk:
        return 'Medium Risk'
    else:
        return 'Low Risk'

product_type['risk_level'] = product_type['product_type_name'].apply(risk_level)


# **3. Product**

**1.Data Understanding**

In [28]:
# Dữ liệu
product.head()

,order_id,product_type_name,total_weight,main_rev
0,156,Thiết Bị Xe/Phụ Tùng,NaN,NaN
1,171,Thiết Bị Xe/Phụ Tùng,NaN,NaN
2,290,Thiết Bị Xe/Phụ Tùng,NaN,NaN
3,372,Thiết Bị Xe/Phụ Tùng,NaN,NaN
4,620,Thiết Bị Xe/Phụ Tùng,NaN,NaN


In [29]:
# Số dòng và cột dữ liệu
print(f"-Số dòng của dữ liệu: {product.shape[0]} dòng.")
print(f"-Số cột của dữ liệu: {product.shape[1]} cột.")

-Số dòng của dữ liệu: 1601 dòng.
-Số cột của dữ liệu: 4 cột.


In [30]:
# Tổng quan thuộc tính dữ liệu
product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1601 entries, 0 to 1600
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           1601 non-null   int64  
 1   product_type_name  1601 non-null   object 
 2   total_weight       1552 non-null   float64
 3   main_rev           1552 non-null   float64
dtypes: float64(2), int64(1), object(1)
memory usage: 50.2+ KB


In [31]:
# Thuộc tính dạng số
numerical_cols_p = product.select_dtypes(include=['number']).columns
print(f"-Numerical Columns: {numerical_cols_p}")

-Numerical Columns: Index(['order_id', 'total_weight', 'main_rev'], dtype='object')


In [32]:
#Thống kê thuộc tính dạng số
product.describe()

,order_id,total_weight,main_rev
count,1601.000000,1.552000e+03,1.552000e+03
mean,1420.407870,3.128707e+12,4.300015e+12
std,818.084538,3.524109e+13,4.323299e+13
min,8.000000,2.500000e-02,7.250000e+03
25%,713.000000,6.000000e-01,1.450000e+05
50%,1426.000000,1.575000e+00,3.688800e+05
75%,2138.000000,1.001750e+01,1.109492e+06
max,2949.000000,7.858333e+14,8.280533e+14


In [33]:
# Thuộc tính dạng phân loại
categorical_cols_p = product.select_dtypes(include= ['object', 'category']).columns
print(f"-Categorical columns: {categorical_cols_p}")

-Categorical columns: Index(['product_type_name'], dtype='object')


In [34]:
# Thống kê thuộc tính dạng phân loại
for col in categorical_cols_p:
    print(f"Phân bổ thuộc tính {col}")
    print(product[col].value_counts(ascending= False))

Phân bổ thuộc tính product_type_name
product_type_name
Thiết Bị Xe/Phụ Tùng         677
Quần Áo                      232
Đồ Chơi                      192
Thực Phẩm                     77
Đồ Trang Trí                  66
Điện tử                       62
Thú Cưng                      56
Mỹ Phẩm                       51
Thuốc/Thực Phẩm Chức Năng     48
Giày dép                      28
Gia Dụng                      24
Văn Phòng Phẩm                23
Phụ Kiện/ Trang Sức           22
Khác                          17
Thể Thao                      11
Mẹ & Bé                       10
Nước hoa                       4
Cigar                          1
Name: count, dtype: int64


**2. Data cleaning**


In [35]:
# Kiểm tra giá trị null
product.isnull().sum()

order_id              0
product_type_name     0
total_weight         49
main_rev             49
dtype: int64

In [36]:
# Fill total_weight với median theo category
product['total_weight'] = product['total_weight'].fillna(product.groupby('product_type_name')['total_weight'].transform('median'))

In [37]:
# Check lại sau khi fill 
product['total_weight'].isnull().sum()

np.int64(0)

In [38]:
# Fill rev cho đơn hàng chưa hoàn thành
product['main_rev']=product['main_rev'].fillna(0)

In [39]:
# Check lại sau khi fill 
product['main_rev'].isnull().sum()

np.int64(0)

In [40]:
#Kiểm tra giá trị trùng lặp
print(f"-Số dòng trùng lặp: {product.duplicated().sum()}")

-Số dòng trùng lặp: 0


**3.Outliers**

In [41]:
# Giá trị ngoại lai của weight
#Q1, Q3, IQR
weight_Q1 = product['total_weight'].quantile(0.25)
weight_Q3 = product['total_weight'].quantile(0.75)
weight_IQR = weight_Q3 - weight_Q1

# Xác định ngưỡng trên, ngưỡng dưới
weight_lower_bound= weight_Q1 - 1.5 * weight_IQR
weight_upper_bound= weight_Q3 + 1.5 * weight_IQR 

# Các giá trị ngoại lai 
weight_outliers = product[(product['total_weight'] < weight_lower_bound)|(product['total_weight'] > weight_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {weight_IQR: .2f}")
print(f"- Ngưỡng dưới: {weight_lower_bound}")
print(f"- Ngưỡng trên: {weight_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(weight_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(weight_outliers)/len(product)*100:.2f}%")

Kết quả: 
- IQR:  8.88
- Ngưỡng dưới: -12.700000000000001
- Ngưỡng trên: 22.82
- Số lượng giá trị ngoại lai: 284
- Tỷ lệ ngoại lai: 17.74%


In [42]:
# Giá trị ngoại lai của rev
#Q1, Q3, IQR
rev_Q1 = product['main_rev'].quantile(0.25)
rev_Q3 = product['main_rev'].quantile(0.75)
rev_IQR = rev_Q3 - rev_Q1

# Xác định ngưỡng trên, ngưỡng dưới
rev_lower_bound= rev_Q1 - 1.5 * rev_IQR
rev_upper_bound= rev_Q3 + 1.5 * rev_IQR 

# Các giá trị ngoại lai 
rev_outliers = product[(product['main_rev'] < rev_lower_bound)|(product['main_rev'] > rev_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {rev_IQR: .2f}")
print(f"- Ngưỡng dưới: {rev_lower_bound}")
print(f"- Ngưỡng trên: {rev_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(rev_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(rev_outliers)/len(product)*100:.2f}%")

Kết quả: 
- IQR:  903350.00
- Ngưỡng dưới: -1212925.0
- Ngưỡng trên: 2400475.0
- Số lượng giá trị ngoại lai: 186
- Tỷ lệ ngoại lai: 11.62%


**4. Feature Engineering**

In [43]:
#efficient score
product['efficiency'] = product['main_rev'] / product['total_weight']

In [44]:
#rev share
product['order_total_rev'] = product.groupby('order_id')['main_rev'].transform('sum')

product['rev_share'] = product['main_rev'] / product['order_total_rev']

In [45]:
#weight share
product['order_total_weight'] = product.groupby('order_id')['total_weight'].transform('sum')

product['weight_share'] = product['total_weight'] / product['order_total_weight']

In [46]:
# Phân khúc sản phẩm theo doanh thu
product['rev_tier'] = pd.qcut(product['main_rev'], 3, labels=['Low','Medium','High'])

In [47]:
# Category diversity per order
product['num_category_per_order'] = product.groupby('order_id')['product_type_name'].transform('nunique')

# **4.Customer Order**

**1. Data Understanding**

In [48]:
# Dữ liệu
print("-Dữ liệu")
customer_order.head()

-Dữ liệu


,order_id,ngày_lên_đơn,customer_id,route_id,customer_name,order_type,country,status,price_ship,date,Recency Days (col),Recency Bucket,ngày_lên_đơn_VN,hour,Order Month,First Order Date,Cohort Month,Cohort Index,revenue,Column
0,1928,1/15/2026 10:13:02 AM,16,6,Linnie,MUA_HO,IDR,DA_GIAO,290000,1/15/2026 12:00:00 AM,31,31-60,1/15/2026 5:13:02 PM,17,2026-01-01,2025-03-12 00:00:00,2025-01-12,1,NaN,NaN
1,1012,12/21/2025 1:36:30 AM,18,6,Fairy,MUA_HO,IDR,DA_GIAO,290000,12/21/2025 12:00:00 AM,2,0-7,12/21/2025 8:36:30 AM,8,2025-01-12,2025-02-12 00:00:00,2025-01-12,0,NaN,NaN
2,1025,12/21/2025 12:08:34 PM,18,6,Fairy,MUA_HO,IDR,DA_GIAO,290000,12/21/2025 12:00:00 AM,2,0-7,12/21/2025 7:08:34 PM,19,2025-01-12,2025-02-12 00:00:00,2025-01-12,0,NaN,NaN
3,670,2025-11-12 01:46:47,18,6,Fairy,MUA_HO,IDR,DA_GIAO,290000,2025-11-12 00:00:00,2,0-7,2025-11-12 08:46:47,8,2025-01-12,2025-02-12 00:00:00,2025-01-12,0,NaN,NaN
4,679,2025-11-12 04:42:19,18,6,Fairy,MUA_HO,IDR,DA_GIAO,290000,2025-11-12 00:00:00,2,0-7,2025-11-12 11:42:19,11,2025-01-12,2025-02-12 00:00:00,2025-01-12,0,NaN,NaN


In [49]:
# Số dòng và cột dữ liệu
print(f"-Số dòng của dữ liệu: {customer_order.shape[0]} dòng.")
print(f"-Số cột của dữ liệu là: {customer_order.shape[1]} cột.")

-Số dòng của dữ liệu: 2986 dòng.
-Số cột của dữ liệu là: 20 cột.


In [50]:
# Tổng quan thuộc tính dữ liệu
customer_order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2986 entries, 0 to 2985
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   order_id            2986 non-null   int64         
 1   ngày_lên_đơn        2986 non-null   object        
 2   customer_id         2986 non-null   int64         
 3   route_id            2986 non-null   int64         
 4   customer_name       2986 non-null   object        
 5   order_type          2986 non-null   object        
 6   country             2986 non-null   object        
 7   status              2986 non-null   object        
 8   price_ship          2986 non-null   int64         
 9   date                2986 non-null   object        
 10  Recency Days (col)  2986 non-null   int64         
 11  Recency Bucket      2986 non-null   object        
 12  ngày_lên_đơn_VN     2986 non-null   object        
 13  hour                2986 non-null   int64       

In [51]:
# Thuộc tính dạng số
numerical_cols_cus = customer_order.select_dtypes(include=['number']).columns
print(f"-Numerical columns: {numerical_cols_cus}")

-Numerical columns: Index(['order_id', 'customer_id', 'route_id', 'price_ship',
       'Recency Days (col)', 'hour', 'Cohort Index', 'revenue', 'Column'],
      dtype='object')


In [52]:
# Thống kê thuộc tính dạng số
customer_order.describe()

,order_id,customer_id,route_id,price_ship,Recency Days (col),hour,Order Month,Cohort Month,Cohort Index,revenue,Column
count,2986.000000,2986.000000,2986.000000,2.986000e+03,2986.000000,2986.000000,2986,2986,2986.000000,0.0,0.0
mean,1616.426658,1005.844608,5.204956,2.266661e+05,24.703617,13.783322,2025-08-03 04:25:43.201607424,2025-03-14 01:03:10.488948480,0.987944,NaN,NaN
min,2.000000,16.000000,1.000000,0.000000e+00,0.000000,0.000000,2025-01-11 00:00:00,2025-01-11 00:00:00,0.000000,NaN,NaN
25%,823.250000,445.000000,6.000000,1.450000e+05,2.000000,10.000000,2025-01-12 00:00:00,2025-01-11 00:00:00,0.000000,NaN,NaN
50%,1624.500000,1106.000000,6.000000,2.900000e+05,8.000000,14.000000,2026-01-01 00:00:00,2025-01-12 00:00:00,1.000000,NaN,NaN
75%,2409.750000,1484.750000,6.000000,2.900000e+05,43.000000,16.000000,2026-01-01 00:00:00,2025-01-12 00:00:00,2.000000,NaN,NaN
max,3161.000000,1944.000000,6.000000,2.900000e+06,107.000000,23.000000,2026-01-03 00:00:00,2026-01-03 00:00:00,4.000000,NaN,NaN
std,906.464163,579.245267,1.565388,1.113128e+05,29.033145,3.949379,NaN,NaN,1.067499,NaN,NaN


In [53]:
# Thuộc tính dạng phân loại
categorical_cols_cus = customer_order.select_dtypes(include=['object', 'category']).columns
print(f"-Categorical columns: {categorical_cols_cus}")

-Categorical columns: Index(['ngày_lên_đơn', 'customer_name', 'order_type', 'country', 'status',
       'date', 'Recency Bucket', 'ngày_lên_đơn_VN', 'First Order Date'],
      dtype='object')


In [54]:
# Thống kê thuộc tính phân loại
for col in categorical_cols_cus:
    print("Phân bổ thuộc tính {col}:")
    print(customer_order[col].value_counts(ascending=False))

Phân bổ thuộc tính {col}:
ngày_lên_đơn
2026-05-02 10:23:29       1
1/15/2026 10:13:02 AM     1
12/21/2025 1:36:30 AM     1
12/21/2025 12:08:34 PM    1
2025-11-12 01:46:47       1
                         ..
12/29/2025 7:23:55 AM     1
11/27/2025 6:46:00 AM     1
12/13/2025 8:01:36 AM     1
12/29/2025 2:48:47 AM     1
12/15/2025 7:24:20 AM     1
Name: count, Length: 2986, dtype: int64
Phân bổ thuộc tính {col}:
customer_name
Nguyen Tien Son            127
Nguyễn Thế Hiếu            123
Đặng Mạnh Hiệp              65
Trần Quốc Phong             55
Vân                         52
                          ... 
Longnguyen                   1
Trần Trọng Hiếu              1
Đinh Hải Nam                 1
Vũ Trưởng - Quang Thanh      1
Phạm Hà Phương               1
Name: count, Length: 658, dtype: int64
Phân bổ thuộc tính {col}:
order_type
MUA_HO         1688
KY_GUI          860
CHUYEN_TIEN     348
DAU_GIA          90
Name: count, dtype: int64
Phân bổ thuộc tính {col}:
country
IDR    2325
JPY 

**2. Data Cleaning**

In [55]:
# Kiểm tra giá trị null
customer_order.isnull().sum()

order_id                 0
ngày_lên_đơn             0
customer_id              0
route_id                 0
customer_name            0
order_type               0
country                  0
status                   0
price_ship               0
date                     0
Recency Days (col)       0
Recency Bucket           0
ngày_lên_đơn_VN          0
hour                     0
Order Month              0
First Order Date         0
Cohort Month             0
Cohort Index             0
revenue               2986
Column                2986
dtype: int64

In [56]:
# merge fact_warehouse lấy rev
customer_order = customer_order.merge(warehouse[['order_id', 'revenue_main']].drop_duplicates(),on='order_id',how='left')
# fill null
customer_order['revenue'] = customer_order['revenue'].fillna(customer_order['revenue_main'])

In [57]:
# check null sau fill
customer_order['revenue'].isnull().sum()

np.int64(1117)

In [58]:
customer_order['order_id'].isin(warehouse['order_id']).value_counts()

order_id
True     2222
False    1117
Name: count, dtype: int64

In [59]:
# Kiểm tra giá trị trùng lặp
print(f"-Số dòng trùng lặp: {customer_order.duplicated().sum()}")

-Số dòng trùng lặp: 0


**3. Outliers**

In [60]:
# Giá trị ngoại lai của price_ship
#Q1, Q3, IQR
ps_Q1 = customer_order['price_ship'].quantile(0.25)
ps_Q3 = customer_order['price_ship'].quantile(0.75)
ps_IQR = ps_Q3 - ps_Q1

# Xác định ngưỡng trên, ngưỡng dưới
ps_lower_bound= ps_Q1 - 1.5 * ps_IQR
ps_upper_bound= ps_Q3 + 1.5 * ps_IQR 

# Các giá trị ngoại lai 
ps_outliers = customer_order[(customer_order['price_ship'] < ps_lower_bound)|(customer_order['price_ship'] > ps_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {ps_IQR: .2f}")
print(f"- Ngưỡng dưới: {ps_lower_bound}")
print(f"- Ngưỡng trên: {ps_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(ps_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(ps_outliers)/len(customer_order)*100:.2f}%")

Kết quả: 
- IQR:  145000.00
- Ngưỡng dưới: -72500.0
- Ngưỡng trên: 507500.0
- Số lượng giá trị ngoại lai: 1
- Tỷ lệ ngoại lai: 0.03%


In [61]:
# Giá trị ngoại lai của revenue
#Q1, Q3, IQR
rev_Q1 = customer_order['revenue'].quantile(0.25)
rev_Q3 = customer_order['revenue'].quantile(0.75)
rev_IQR = rev_Q3 - rev_Q1

# Xác định ngưỡng trên, ngưỡng dưới
rev_lower_bound= rev_Q1 - 1.5 * rev_IQR
rev_upper_bound= rev_Q3 + 1.5 * rev_IQR 

# Các giá trị ngoại lai 
rev_outliers = customer_order[(customer_order['revenue'] < rev_lower_bound)|(customer_order['revenue'] > rev_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {rev_IQR: .2f}")
print(f"- Ngưỡng dưới: {rev_lower_bound}")
print(f"- Ngưỡng trên: {rev_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(rev_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(rev_outliers)/len(customer_order)*100:.2f}%")

Kết quả: 
- IQR:  629285.00
- Ngưỡng dưới: -653927.5
- Ngưỡng trên: 1863212.5
- Số lượng giá trị ngoại lai: 311
- Tỷ lệ ngoại lai: 9.31%


**4. Feature Engineering**

In [62]:
# Tạo flag cho các đơn đã hoàn thành
customer_order['is_completed'] = customer_order['order_id'].isin(warehouse['order_id'])

In [63]:
#Bỏ cột nghiệp vụ không cần thiết
customer_order.drop(columns=['Column'], inplace=True)

In [64]:
# Time standardization
customer_order['ngày_lên_đơn'] = pd.to_datetime(customer_order['ngày_lên_đơn'])
customer_order['ngày_lên_đơn_VN'] = pd.to_datetime(customer_order['ngày_lên_đơn_VN'])
customer_order['date'] = pd.to_datetime(customer_order['date'])
customer_order['First Order Date'] = pd.to_datetime(customer_order['First Order Date'])
# Time featuring
customer_order['day_of_week'] = customer_order['ngày_lên_đơn_VN'].dt.day_name()
customer_order['day_of_week'] = customer_order['ngày_lên_đơn_VN'].dt.day_name()
#Time bucket
def time_bucket(h):
    if 6 <= h < 12:
        return 'Morning'
    elif 12 <= h < 18:
        return 'Afternoon'
    elif 18 <= h < 23:
        return 'Evening'
    else:
        return 'Night'

customer_order['time_bucket'] = customer_order['hour'].apply(time_bucket)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3708\3692835402.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  customer_order['ngày_lên_đơn_VN'] = pd.to_datetime(customer_order['ngày_lên_đơn_VN'])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_3708\3692835402.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  customer_order['date'] = pd.to_datetime(customer_order['date'])


In [65]:
# Phân khúc khách hàng
customer_order['order_value_segment'] = pd.qcut(
    customer_order['revenue'],
    3,
    labels=['Low','Medium','High']
)
# Tổng số đơn per khách hàng
customer_order['total_order'] = customer_order.groupby('customer_id')['order_id'].transform('count')
# Customer lifetime valur
customer_order['CLV'] = customer_order.groupby('customer_id')['revenue'].transform('sum')
# repeat customer
customer_order['is_repeated'] = customer_order['total_order'] > 1

In [66]:
# country mapping
customer_order['market'] = customer_order['country'].map({'USD':'US','JPY':'Japan','KRW':'Korea','IDR':'Indonesia'})

In [67]:
# advanced feature
customer_order['is_returning_cohort'] = customer_order['Cohort Index'] > 0

# **5. Link**

**1. Data Understanding**

In [68]:
# Dữ liệu
print('Dữ liệu:')
link.head()

Dữ liệu:


,link_id,product_name,product_type_name,name,total_web,exchange_rate,final_price_vnd,final_price,forex_margin,order_id,created_at,date,customer,First purchase,Cohort Month reve,Order Month,Cohort Index
0,1121,Motorcycle spare parts,Thiết Bị Xe/Phụ Tùng,IDR,NaN,1.68,0.0,NaN,NaN,814,12/15/2025 3:09:11 AM,12/15/2025 12:00:00 AM,1268,11/27/2025 12:00:00 AM,2025-01-11,2025-01-12,1
1,1211,Motorcycle spare parts,Thiết Bị Xe/Phụ Tùng,IDR,NaN,1.68,0.0,NaN,NaN,881,12/17/2025 2:56:03 AM,12/17/2025 12:00:00 AM,1268,11/27/2025 12:00:00 AM,2025-01-11,2025-01-12,1
2,1358,Motorcycle spare parts,Thiết Bị Xe/Phụ Tùng,IDR,NaN,1.68,0.0,NaN,NaN,984,12/19/2025 10:53:44 AM,12/19/2025 12:00:00 AM,1136,11/28/2025 12:00:00 AM,2025-01-11,2025-01-12,1
3,1438,Motorcycle spare parts,Thiết Bị Xe/Phụ Tùng,IDR,NaN,1.68,0.0,NaN,NaN,1052,12/22/2025 4:43:12 AM,12/22/2025 12:00:00 AM,1193,11/24/2025 12:00:00 AM,2025-01-11,2025-01-12,1
4,1439,Motorcycle spare parts,Thiết Bị Xe/Phụ Tùng,IDR,NaN,1.68,0.0,NaN,NaN,1053,12/22/2025 4:43:42 AM,12/22/2025 12:00:00 AM,1193,11/24/2025 12:00:00 AM,2025-01-11,2025-01-12,1


In [69]:
# Số dòng và số cột dữ liệu
print(f"-Số dòng của dữ liệu: {link.shape[0]} dòng.")
print(f"-Số cột của dữ liệu: {link.shape[1]} cột.")

-Số dòng của dữ liệu: 4798 dòng.
-Số cột của dữ liệu: 17 cột.


In [70]:
# Tổng quan thuộc tính dữ liệu
link.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4798 entries, 0 to 4797
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   link_id            4798 non-null   int64         
 1   product_name       4798 non-null   object        
 2   product_type_name  4704 non-null   object        
 3   name               4798 non-null   object        
 4   total_web          3665 non-null   float64       
 5   exchange_rate      4798 non-null   float64       
 6   final_price_vnd    4798 non-null   float64       
 7   final_price        3665 non-null   float64       
 8   forex_margin       3665 non-null   float64       
 9   order_id           4798 non-null   int64         
 10  created_at         4798 non-null   object        
 11  date               4798 non-null   object        
 12  customer           4798 non-null   int64         
 13  First purchase     4798 non-null   object        
 14  Cohort M

In [71]:
# Thuộc tính dạng số
numerical_cols_l = link.select_dtypes(include=['number']).columns
print(f"-Numerical columns: {numerical_cols_l}")

-Numerical columns: Index(['link_id', 'total_web', 'exchange_rate', 'final_price_vnd',
       'final_price', 'forex_margin', 'order_id', 'customer', 'Cohort Index'],
      dtype='object')


In [72]:
# Thống kê thuộc tính dạng số
link.describe()

,link_id,total_web,exchange_rate,final_price_vnd,final_price,forex_margin,order_id,customer,Cohort Month reve,Order Month,Cohort Index
count,4798.000000,3.665000e+03,4798.000000,4.798000e+03,3.665000e+03,3.665000e+03,4798.000000,4798.000000,4798,4798,4798.000000
mean,2598.157357,8.546892e+05,738.938958,2.322852e+06,3.000351e+06,3.879686e+04,1735.176115,1037.138391,2025-03-29 06:47:52.196748544,2025-08-24 16:14:12.355148032,1.027928
min,2.000000,1.000000e+00,1.680000,0.000000e+00,1.680000e+00,-2.504975e+06,2.000000,16.000000,2025-01-11 00:00:00,2025-01-11 00:00:00,0.000000
25%,1318.250000,4.010000e+04,1.680000,3.400000e+04,1.764000e+05,1.281000e+03,953.250000,445.000000,2025-01-11 00:00:00,2025-01-12 00:00:00,0.000000
50%,2624.500000,1.652800e+05,1.680000,2.686000e+05,4.418000e+05,4.970000e+03,1783.500000,1192.000000,2025-01-12 00:00:00,2026-01-01 00:00:00,1.000000
75%,3884.750000,5.177000e+05,1.680000,8.696650e+05,1.277640e+06,1.664000e+04,2514.000000,1504.000000,2025-01-12 00:00:00,2026-01-02 00:00:00,2.000000
max,5105.000000,6.577400e+07,28000.000000,2.461534e+09,2.435440e+09,2.609400e+07,3161.000000,1944.000000,2026-01-03 00:00:00,2026-01-03 00:00:00,4.000000
std,1468.860735,3.087868e+06,4404.264446,4.252235e+07,4.826653e+07,5.087542e+05,911.614495,584.507453,NaN,NaN,1.120642


In [73]:
# Thuộc tính dạng phân loại
categorical_cols_l = link.select_dtypes(include=['object', 'category']).columns
print(f"-Cateogical columns: {categorical_cols_l}")

-Cateogical columns: Index(['product_name', 'product_type_name', 'name', 'created_at', 'date',
       'First purchase'],
      dtype='object')


In [74]:
# Thống kê thuộc tính phân loại
for col in categorical_cols_l:
    print("Phân bổ thuộc tính {col}:")
    print(link[col].value_counts(ascending=False))

Phân bổ thuộc tính {col}:
product_name
Money Exchange            633
Motorbike spare parts     459
Auto Part                 363
Motorcycle spare parts    347
Clothes                   173
                         ... 
children's toys             1
toy model                   1
Thiết bị may                1
Tư bản                      1
khăn                        1
Name: count, Length: 520, dtype: int64
Phân bổ thuộc tính {col}:
product_type_name
Thiết Bị Xe/Phụ Tùng         1797
Tài Chính                     617
Quần Áo                       597
Đồ Chơi                       490
Thực Phẩm                     184
Thú Cưng                      160
Đồ Trang Trí                  153
Mỹ Phẩm                       139
Văn Phòng Phẩm                102
Điện tử                        98
Thuốc/Thực Phẩm Chức Năng      90
Gia Dụng                       81
Khác                           47
Phụ Kiện/ Trang Sức            41
Giày dép                       37
Mẹ & Bé                        23
Ciga

**2. Data cleaning**

In [75]:
# kiểm tra giá trị null
link.isnull().sum()

link_id                 0
product_name            0
product_type_name      94
name                    0
total_web            1133
exchange_rate           0
final_price_vnd         0
final_price          1133
forex_margin         1133
order_id                0
created_at              0
date                    0
customer                0
First purchase          0
Cohort Month reve       0
Order Month             0
Cohort Index            0
dtype: int64

In [76]:
#Fill null
link['product_type_name'] = link['product_type_name'].fillna('Unknown')
#1133 order thuộc loại kí gửi hoặc chuyển tiền nên các thuộc tính bên trên null

In [78]:
#Kiểm tra giá trị trùng lặp
print(f"-Số dòng dữ liệu trùng lặp {link.duplicated().sum()}")

-Số dòng dữ liệu trùng lặp 0


**3. Outliers**

In [82]:
# Giá trị ngoại lai của final_price_vnd
#Q1, Q3, IQR
pricev_Q1 = link['final_price_vnd'].quantile(0.25)
pricev_Q3 = link['final_price_vnd'].quantile(0.75)
pricev_IQR = pricev_Q3 - pricev_Q1

# Xác định ngưỡng trên, ngưỡng dưới
pricev_lower_bound= pricev_Q1 - 1.5 * pricev_IQR
pricev_upper_bound= pricev_Q3 + 1.5 * pricev_IQR 

# Các giá trị ngoại lai 
pricev_outliers = link[(link['final_price_vnd'] < pricev_lower_bound)|(link['final_price_vnd'] > pricev_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {pricev_IQR: .2f}")
print(f"- Ngưỡng dưới: {pricev_lower_bound}")
print(f"- Ngưỡng trên: {pricev_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(pricev_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(pricev_outliers)/len(link)*100:.2f}%")

Kết quả: 
- IQR:  835665.00
- Ngưỡng dưới: -1219497.5
- Ngưỡng trên: 2123162.5
- Số lượng giá trị ngoại lai: 639
- Tỷ lệ ngoại lai: 13.32%


In [83]:
# Giá trị ngoại lai của final_price
#Q1, Q3, IQR
price_Q1 = link['final_price'].quantile(0.25)
price_Q3 = link['final_price'].quantile(0.75)
price_IQR = price_Q3 - price_Q1

# Xác định ngưỡng trên, ngưỡng dưới
price_lower_bound= price_Q1 - 1.5 * price_IQR
price_upper_bound= price_Q3 + 1.5 * price_IQR 

# Các giá trị ngoại lai 
price_outliers = link[(link['final_price'] < price_lower_bound)|(link['final_price'] > price_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {price_IQR: .2f}")
print(f"- Ngưỡng dưới: {price_lower_bound}")
print(f"- Ngưỡng trên: {price_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(price_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(price_outliers)/len(link)*100:.2f}%")

Kết quả: 
- IQR:  1101240.00
- Ngưỡng dưới: -1475460.0
- Ngưỡng trên: 2929500.0
- Số lượng giá trị ngoại lai: 478
- Tỷ lệ ngoại lai: 9.96%


**4. Feature Enngineering**

In [88]:
# Time standardization
link['created_at'] = pd.to_datetime(link['created_at'])
link['date'] = pd.to_datetime(link['date'])
link['First purchase'] = pd.to_datetime(link['First purchase'])

C:\Users\ASUS\AppData\Local\Temp\ipykernel_3708\3726566034.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  link['date'] = pd.to_datetime(link['date'])
C:\Users\ASUS\AppData\Local\Temp\ipykernel_3708\3726566034.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  link['First purchase'] = pd.to_datetime(link['First purchase'])


In [84]:
# Đơn hàng có liên quan đến mua bán không?
link['is_commercial'] = link['final_price_vnd'] > 0

In [85]:
# margin rate
link['margin_rate'] = link['forex_margin'] / link['final_price_vnd']
# margin on cost
link['margin_on_cost'] = link['forex_margin'] / link['final_price']

In [86]:
#price segmentaion
link['price_segment'] = pd.qcut(link['final_price_vnd'],4,labels=['Low','Mid-low','Mid-high','High'])

In [89]:
# Recency
link['days_since_first_purchase'] = (link['date'] - link['First purchase']).dt.days

In [91]:
#new vs returning customer
link['is_new_customer'] = link['Cohort Index'] == 0

In [92]:
# items per order
link['items_per_order'] = link.groupby('order_id')['link_id'].transform('count')
# order value
link['order_value'] = link.groupby('order_id')['final_price_vnd'].transform('sum')
# item share
link['item_share'] = link['final_price_vnd'] / link['order_value']

# **6. Warehouse**

**1. Data Understanding**

In [93]:
# Dữ liệu
print("-Dữ liệu:")
warehouse.head()

-Dữ liệu:


,warehouse_id,order_id,route_id,name,ngay_nhap_kho_nn,ngay_xuat_kho_nn,ngay_nhap_kho_vn,ngay_giao_vnpost,net_weight,price_ship,...,product_type_name,n_types,share,final_weight_allocated,revenue_allocated,date,customer_id,thời gian nằm kho nn,thời gian nằm kho vn,thời gian bay
0,2608,3089,6,IDR,2026-05-03 10:04:20,NaN,NaN,NaN,0.675,290000,...,Quần Áo,1,1.0,1.0,290000.0,2026-05-03 00:00:00,25,NaN,NaN,NaN
1,2641,3103,6,IDR,2026-05-03 11:57:22,NaN,NaN,NaN,0.800,290000,...,Đồ Trang Trí,1,1.0,1.0,290000.0,2026-05-03 00:00:00,28,NaN,NaN,NaN
2,2642,3103,6,IDR,2026-05-03 12:02:00,NaN,NaN,NaN,0.490,290000,...,Đồ Trang Trí,1,1.0,1.0,290000.0,2026-05-03 00:00:00,28,NaN,NaN,NaN
3,2643,3103,6,IDR,2026-05-03 12:04:15,NaN,NaN,NaN,0.400,290000,...,Đồ Trang Trí,1,1.0,1.0,290000.0,2026-05-03 00:00:00,28,NaN,NaN,NaN
4,2680,3110,6,IDR,2026-07-03 02:14:27,NaN,NaN,NaN,0.700,290000,...,Quần Áo,1,1.0,1.0,290000.0,2026-07-03 00:00:00,41,NaN,NaN,NaN


In [94]:
# Số dòng và cột dự liệu
print(f"-Số dòng của dữ liệu: {warehouse.shape[0]} dòng.")
print(f"-Số cột của dữ liệu: {warehouse.shape[1]} cột.")

-Số dòng của dữ liệu: 2789 dòng.
-Số cột của dữ liệu: 24 cột.


In [95]:
# Tổng quan thuộc tính dữ liệu
warehouse.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2789 entries, 0 to 2788
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   warehouse_id            2789 non-null   int64  
 1   order_id                2789 non-null   int64  
 2   route_id                2789 non-null   int64  
 3   name                    2789 non-null   object 
 4   ngay_nhap_kho_nn        2789 non-null   object 
 5   ngay_xuat_kho_nn        2518 non-null   object 
 6   ngay_nhap_kho_vn        2495 non-null   object 
 7   ngay_giao_vnpost        2142 non-null   object 
 8   net_weight              2789 non-null   float64
 9   price_ship              2789 non-null   int64  
 10  min_weight              2789 non-null   float64
 11  final_weight            2789 non-null   float64
 12  revenue_main            2789 non-null   float64
 13  product_type_id         2789 non-null   int64  
 14  product_type_name       2789 non-null   

In [96]:
# Thuộc tính dạng số
numerical_cols_w = warehouse.select_dtypes(include=['number']).columns
print(f"-Numerical columns: {numerical_cols_w}")

-Numerical columns: Index(['warehouse_id', 'order_id', 'route_id', 'net_weight', 'price_ship',
       'min_weight', 'final_weight', 'revenue_main', 'product_type_id',
       'n_types', 'share', 'final_weight_allocated', 'revenue_allocated',
       'customer_id', 'thời gian nằm kho nn', 'thời gian nằm kho vn',
       'thời gian bay'],
      dtype='object')


In [97]:
# Thống kê thuộc tính dạng số
warehouse.describe()

,warehouse_id,order_id,route_id,net_weight,price_ship,min_weight,final_weight,revenue_main,product_type_id,n_types,share,final_weight_allocated,revenue_allocated,customer_id,thời gian nằm kho nn,thời gian nằm kho vn,thời gian bay
count,2789.000000,2789.000000,2789.000000,2.789000e+03,2789.000000,2789.000000,2.789000e+03,2.789000e+03,2789.000000,2789.000000,2789.000000,2.789000e+03,2.789000e+03,2789.000000,2518.000000,2112.000000,2495.000000
mean,1375.413768,1688.337755,5.109717,1.912364e+12,255242.724274,0.915023,1.912364e+12,1.480912e+12,13.403012,1.113302,0.939441,1.527519e+12,2.277600e+13,1068.499104,2.720413,11.482008,3.426453
std,781.506457,850.826082,1.629449,2.936766e+13,61017.474083,0.187830,2.936766e+13,2.593142e+13,4.226590,0.421872,0.219260,2.563545e+13,1.324212e+14,592.768937,3.485663,11.470542,2.146056
min,3.000000,8.000000,1.000000,1.000000e-02,2000.000000,0.500000,5.000000e-01,2.000000e+03,1.000000,1.000000,0.066667,3.333333e-02,2.000000e+03,16.000000,0.000000,0.000000,0.000000
25%,693.000000,973.000000,6.000000,4.000000e-01,280000.000000,1.000000,1.000000e+00,2.900000e+05,12.000000,1.000000,1.000000,1.000000e+00,2.900000e+05,692.000000,0.000000,3.000000,2.000000
50%,1416.000000,1761.000000,6.000000,1.000000e+00,290000.000000,1.000000,1.000000e+00,2.900000e+05,12.000000,1.000000,1.000000,1.000000e+00,2.900000e+05,1239.000000,1.000000,6.000000,4.000000
75%,2035.000000,2377.000000,6.000000,4.400000e+00,290000.000000,1.000000,4.400000e+00,7.203600e+05,16.000000,1.000000,1.000000,4.100000e+00,8.120000e+05,1552.000000,4.000000,19.000000,5.000000
max,2712.000000,3154.000000,6.000000,7.858333e+14,500000.000000,1.000000,7.858333e+14,7.443333e+14,24.000000,3.000000,1.000000,7.858333e+14,9.763333e+14,1931.000000,42.000000,68.000000,29.000000


In [98]:
# Thuộc tính dạng phân loại 
categorical_cols_w = warehouse.select_dtypes(include=['object','category']).columns
print(f"-Categorical colomns: {categorical_cols_w}")

-Categorical colomns: Index(['name', 'ngay_nhap_kho_nn', 'ngay_xuat_kho_nn', 'ngay_nhap_kho_vn',
       'ngay_giao_vnpost', 'product_type_name', 'date'],
      dtype='object')


In [99]:
# Thống kê thuộc tính phân loại
for col in categorical_cols_w:
    print("Phân bổ thuộc tính {col}:")
    print(warehouse[col].value_counts(ascending=False))

Phân bổ thuộc tính {col}:
name
IDR    2107
JPY     474
KRW     151
USD      57
Name: count, dtype: int64
Phân bổ thuộc tính {col}:
ngay_nhap_kho_nn
1/26/2026 10:52:34 AM    3
1/30/2026 8:34:40 AM     3
2026-03-02 08:50:53      3
1/30/2026 8:32:25 AM     3
1/30/2026 8:56:00 AM     3
                        ..
2026-06-03 04:10:46      1
2026-05-03 07:59:13      1
2026-05-03 09:04:55      1
2025-08-12 05:22:45      1
1/26/2026 4:40:26 AM     1
Name: count, Length: 2667, dtype: int64
Phân bổ thuộc tính {col}:
ngay_xuat_kho_nn
2026-03-03 02:13:10      141
2/24/2026 6:32:19 AM     131
2026-02-01 12:20:54       55
1/15/2026 4:42:46 AM      52
1/30/2026 9:24:58 AM      51
                        ... 
12/22/2025 2:32:23 AM      1
12/22/2025 2:36:32 AM      1
2026-03-01 10:00:45        1
12/21/2025 2:06:17 PM      1
2025-04-12 07:26:11        1
Name: count, Length: 356, dtype: int64
Phân bổ thuộc tính {col}:
ngay_nhap_kho_vn
1/19/2026 10:39:57 AM    212
12/18/2025 7:02:39 AM    178
2026-03-02 02

**2. Data Cleaning**



In [ ]:
# Kiểm tra giá trị null
warehouse.isnull().sum()
# Các thuộc tính thời gian có null là do order_type khác nhau và order processing

warehouse_id                0
order_id                    0
route_id                    0
name                        0
ngay_nhap_kho_nn            0
ngay_xuat_kho_nn          271
ngay_nhap_kho_vn          294
ngay_giao_vnpost          647
net_weight                  0
price_ship                  0
min_weight                  0
final_weight                0
revenue_main                0
product_type_id             0
product_type_name           0
n_types                     0
share                       0
final_weight_allocated      0
revenue_allocated           0
date                        0
customer_id                 0
thời gian nằm kho nn      271
thời gian nằm kho vn      677
thời gian bay             294
dtype: int64

In [101]:
#Kiểm tra giá trị trùng lặp
print(f"Số dòng dữ liệu trùng lặp: {warehouse.duplicated().sum()}")

Số dòng dữ liệu trùng lặp: 0


**3.Outliers**

In [102]:
# Giá trị ngoại lai của net_weight
#Q1, Q3, IQR
nw_Q1 = warehouse['net_weight'].quantile(0.25)
nw_Q3 = warehouse['net_weight'].quantile(0.75)
nw_IQR = nw_Q3 - nw_Q1

# Xác định ngưỡng trên, ngưỡng dưới
nw_lower_bound= nw_Q1 - 1.5 * nw_IQR
nw_upper_bound= nw_Q3 + 1.5 * nw_IQR 

# Các giá trị ngoại lai 
nw_outliers = warehouse[(warehouse['net_weight'] < nw_lower_bound)|(warehouse['net_weight'] > nw_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {nw_IQR: .2f}")
print(f"- Ngưỡng dưới: {nw_lower_bound}")
print(f"- Ngưỡng trên: {nw_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(nw_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(nw_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  4.00
- Ngưỡng dưới: -5.6
- Ngưỡng trên: 10.4
- Số lượng giá trị ngoại lai: 488
- Tỷ lệ ngoại lai: 17.50%


In [103]:
# Giá trị ngoại lai của final_weight
#Q1, Q3, IQR
fw_Q1 = warehouse['final_weight'].quantile(0.25)
fw_Q3 = warehouse['final_weight'].quantile(0.75)
fw_IQR = fw_Q3 - fw_Q1

# Xác định ngưỡng trên, ngưỡng dưới
fw_lower_bound= fw_Q1 - 1.5 * fw_IQR
fw_upper_bound= fw_Q3 + 1.5 * fw_IQR 

# Các giá trị ngoại lai 
fw_outliers = warehouse[(warehouse['final_weight'] < fw_lower_bound)|(warehouse['final_weight'] > fw_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {fw_IQR: .2f}")
print(f"- Ngưỡng dưới: {fw_lower_bound}")
print(f"- Ngưỡng trên: {fw_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(fw_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(fw_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  3.40
- Ngưỡng dưới: -4.1000000000000005
- Ngưỡng trên: 9.5
- Số lượng giá trị ngoại lai: 513
- Tỷ lệ ngoại lai: 18.39%


In [104]:
# Giá trị ngoại lai của final_weight_allocated
#Q1, Q3, IQR
fwa_Q1 = warehouse['final_weight_allocated'].quantile(0.25)
fwa_Q3 = warehouse['final_weight_allocated'].quantile(0.75)
fwa_IQR = fwa_Q3 - fwa_Q1

# Xác định ngưỡng trên, ngưỡng dưới
fwa_lower_bound= fwa_Q1 - 1.5 * fwa_IQR
fwa_upper_bound= fwa_Q3 + 1.5 * fwa_IQR 

# Các giá trị ngoại lai 
fwa_outliers = warehouse[(warehouse['final_weight_allocated'] < fwa_lower_bound)|(warehouse['final_weight_allocated'] > fwa_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {fwa_IQR: .2f}")
print(f"- Ngưỡng dưới: {fwa_lower_bound}")
print(f"- Ngưỡng trên: {fwa_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(fwa_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(fwa_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  3.10
- Ngưỡng dưới: -3.6499999999999995
- Ngưỡng trên: 8.75
- Số lượng giá trị ngoại lai: 526
- Tỷ lệ ngoại lai: 18.86%


In [105]:
# Giá trị ngoại lai của revenue_main
#Q1, Q3, IQR
r_Q1 = warehouse['revenue_main'].quantile(0.25)
r_Q3 = warehouse['revenue_main'].quantile(0.75)
r_IQR = r_Q3 - r_Q1

# Xác định ngưỡng trên, ngưỡng dưới
r_lower_bound= r_Q1 - 1.5 * r_IQR
r_upper_bound= r_Q3 + 1.5 * r_IQR 

# Các giá trị ngoại lai 
r_outliers = warehouse[(warehouse['revenue_main'] < r_lower_bound)|(warehouse['revenue_main'] > r_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {r_IQR: .2f}")
print(f"- Ngưỡng dưới: {r_lower_bound}")
print(f"- Ngưỡng trên: {r_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(r_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(r_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  430360.00
- Ngưỡng dưới: -355540.0
- Ngưỡng trên: 1365900.0
- Số lượng giá trị ngoại lai: 426
- Tỷ lệ ngoại lai: 15.27%


In [106]:
# Giá trị ngoại lai của revenue_allocated
#Q1, Q3, IQR
ra_Q1 = warehouse['revenue_allocated'].quantile(0.25)
ra_Q3 = warehouse['revenue_allocated'].quantile(0.75)
ra_IQR = ra_Q3 - ra_Q1

# Xác định ngưỡng trên, ngưỡng dưới
ra_lower_bound= ra_Q1 - 1.5 * ra_IQR
ra_upper_bound= ra_Q3 + 1.5 * ra_IQR 

# Các giá trị ngoại lai 
ra_outliers = warehouse[(warehouse['revenue_allocated'] < ra_lower_bound)|(warehouse['revenue_allocated'] > ra_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {ra_IQR: .2f}")
print(f"- Ngưỡng dưới: {ra_lower_bound}")
print(f"- Ngưỡng trên: {ra_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(ra_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(ra_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  522000.00
- Ngưỡng dưới: -493000.0
- Ngưỡng trên: 1595000.0
- Số lượng giá trị ngoại lai: 440
- Tỷ lệ ngoại lai: 15.78%


In [107]:
# Giá trị ngoại lai của thời gian nằm kho nn
#Q1, Q3, IQR
nn_Q1 = warehouse['thời gian nằm kho nn'].quantile(0.25)
nn_Q3 = warehouse['thời gian nằm kho nn'].quantile(0.75)
nn_IQR = nn_Q3 - nn_Q1

# Xác định ngưỡng trên, ngưỡng dưới
nn_lower_bound= nn_Q1 - 1.5 * nn_IQR
nn_upper_bound= nn_Q3 + 1.5 * nn_IQR 

# Các giá trị ngoại lai 
nn_outliers = warehouse[(warehouse['thời gian nằm kho nn'] < nn_lower_bound)|(warehouse['thời gian nằm kho nn'] > nn_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {nn_IQR: .2f}")
print(f"- Ngưỡng dưới: {nn_lower_bound}")
print(f"- Ngưỡng trên: {nn_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(nn_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(nn_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  4.00
- Ngưỡng dưới: -6.0
- Ngưỡng trên: 10.0
- Số lượng giá trị ngoại lai: 83
- Tỷ lệ ngoại lai: 2.98%


In [108]:
# Giá trị ngoại lai của thời gian nằm kho vn
#Q1, Q3, IQR
vn_Q1 = warehouse['thời gian nằm kho vn'].quantile(0.25)
vn_Q3 = warehouse['thời gian nằm kho vn'].quantile(0.75)
vn_IQR = vn_Q3 - vn_Q1

# Xác định ngưỡng trên, ngưỡng dưới
vn_lower_bound= vn_Q1 - 1.5 * vn_IQR
vn_upper_bound= vn_Q3 + 1.5 * vn_IQR 

# Các giá trị ngoại lai 
vn_outliers = warehouse[(warehouse['thời gian nằm kho vn'] < vn_lower_bound)|(warehouse['thời gian nằm kho vn'] > vn_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {vn_IQR: .2f}")
print(f"- Ngưỡng dưới: {vn_lower_bound}")
print(f"- Ngưỡng trên: {vn_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(vn_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(vn_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  16.00
- Ngưỡng dưới: -21.0
- Ngưỡng trên: 43.0
- Số lượng giá trị ngoại lai: 31
- Tỷ lệ ngoại lai: 1.11%


In [109]:
# Giá trị ngoại lai của thời gian bay
#Q1, Q3, IQR
ft_Q1 = warehouse['thời gian bay'].quantile(0.25)
ft_Q3 = warehouse['thời gian bay'].quantile(0.75)
ft_IQR = ft_Q3 - ft_Q1

# Xác định ngưỡng trên, ngưỡng dưới
ft_lower_bound= ft_Q1 - 1.5 * ft_IQR
ft_upper_bound= ft_Q3 + 1.5 * ft_IQR 

# Các giá trị ngoại lai 
ft_outliers = warehouse[(warehouse['thời gian bay'] < ft_lower_bound)|(warehouse['thời gian bay'] > ft_upper_bound)]

#Kết quả
print("Kết quả: ")
print(f"- IQR: {ft_IQR: .2f}")
print(f"- Ngưỡng dưới: {ft_lower_bound}")
print(f"- Ngưỡng trên: {ft_upper_bound}")
print(f"- Số lượng giá trị ngoại lai: {len(ft_outliers)}")
print(f"- Tỷ lệ ngoại lai: {len(ft_outliers)/len(warehouse)*100:.2f}%")

Kết quả: 
- IQR:  3.00
- Ngưỡng dưới: -2.5
- Ngưỡng trên: 9.5
- Số lượng giá trị ngoại lai: 5
- Tỷ lệ ngoại lai: 0.18%


**4. Feature Engineering**

In [111]:
#Logistic Feature 
warehouse['total_time'] = warehouse['thời gian nằm kho nn'] + warehouse['thời gian nằm kho vn'] + warehouse['thời gian bay']
#Stage contribution
warehouse['pct_warehouse_nn'] = warehouse['thời gian nằm kho nn'] / warehouse['total_time']
warehouse['pct_warehouse_vn'] = warehouse['thời gian nằm kho vn'] / warehouse['total_time']
warehouse['pct_air_time'] = warehouse['thời gian bay'] / warehouse['total_time']
# Delay flag
warehouse['is_delayed'] = warehouse['total_time'] > warehouse['total_time'].median()


In [113]:
#Customer Feature
warehouse['shipment_count'] = warehouse.groupby('customer_id')['order_id'].transform('count')
warehouse['avg_lead_time_customer'] = warehouse.groupby('customer_id')['total_time'].transform('mean')

In [115]:
# Advanced Feature
warehouse['is_complex_order'] = warehouse['n_types'] > 1
warehouse['efficiency_score'] = warehouse['revenue_main'] / warehouse['total_time']